In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

model = "llama-3.3-70b-versatile"

In [ ]:
SYSTEM_PROMPT = """
You are a helpful personal assistant.

Use your tools when you need real data.

If a tool returns an error:
- Do not crash.
- Use the error information to decide what to do next.
- Try a different approach if possible.
- If no alternative is available, explain the limitation to the user.

Give the user a concise final answer.
"""

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]

In [ ]:
def check_calendar(date):
    return f"Calendar for {date}: 10am Standup, 2pm Dentist"

In [ ]:
def flaky_tool(query):
    raise Exception("Service unavailable")

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_calendar",
            "description": "Check calendar events for a given date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {
                        "type": "string"
                    }
                },
                "required": ["date"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "flaky_tool",
            "description": "Search for additional information. This service may sometimes be unavailable.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

In [ ]:
def execute_tool(name, args):

    try:

        if name == "check_calendar":
            return check_calendar(**args)

        if name == "flaky_tool":
            return flaky_tool(**args)

        return f"Unknown tool: {name}"

    except Exception as e:

        return (
            f"Error: {str(e)}. "
            "Try a different approach."
        )

In [ ]:
result = execute_tool(
    "flaky_tool",
    {
        "query": "Find some information"
    }
)

print(result)

In [ ]:
MAX_ITERATIONS = 10

In [ ]:
def run_agent(messages):

    for iteration in range(MAX_ITERATIONS):

        print(
            f"Iteration {iteration + 1}/{MAX_ITERATIONS}"
        )

        # Protect the LLM/API call
        try:

            response = client.chat.completions.create(
                model=model,
                messages=messages,
                tools=tools
            )

        except Exception as e:

            return (
                f"Agent API error: {str(e)}"
            )

        finish_reason = response.choices[0].finish_reason
        msg = response.choices[0].message

        messages.append(msg)

        # Agent finished normally
        if finish_reason == "stop":
            return msg.content

        # Agent requested tools
        if finish_reason == "tool_calls":

            for tc in msg.tool_calls:

                name = tc.function.name

                # Protect argument parsing
                try:

                    args = json.loads(
                        tc.function.arguments
                    )

                except json.JSONDecodeError:

                    args = {}

                print(
                    f"Action: {name}({args})"
                )

                # execute_tool() handles tool crashes
                result = execute_tool(
                    name,
                    args
                )

                print(
                    f"Observation: {result}"
                )

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })

    # If we reach here, MAX_ITERATIONS was reached

    messages.append({
        "role": "user",
        "content": """
        You've reached the maximum number of iterations.
        Give your best final answer using the information
        you have available.
        Do not call any more tools.
        """
    })

    # Generate final answer without tools
    try:

        final = client.chat.completions.create(
            model=model,
            messages=messages
        )

        return final.choices[0].message.content

    except Exception as e:

        return (
            f"Unable to generate final answer: {str(e)}"
        )

In [ ]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": "What's on my calendar today?"
    }
]

answer = run_agent(messages)

print("\nFinal Answer:")
print(answer)

In [ ]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": """
        Use the additional information service
        to search for today's latest information.
        """
    }
]

answer = run_agent(messages)

print("\nFinal Answer:")
print(answer)